# MCP Server Smoke Test

This notebook validates your MCP server over **streamable HTTP**.

It covers:
- initialize
- notifications/initialized
- tools/list
- resources/read (`skg://catalog/tools`)
- tools/call (`lexical_search`)


In [1]:
import json
import os
import subprocess
import time
from dataclasses import dataclass
from typing import Any

import httpx


In [2]:
DEFAULT_ENDPOINT = "http://127.0.0.1:8000/mcp"
DEFAULT_PROTOCOL_VERSION = "2025-03-26"

@dataclass
class TestResult:
    name: str
    passed: bool
    details: str

class MCPStreamableHTTPClient:
    def __init__(self, endpoint: str, timeout_seconds: float = 30.0) -> None:
        self.endpoint = endpoint
        self.session_id: str | None = None
        self.client = httpx.Client(timeout=timeout_seconds)

    def close(self) -> None:
        self.client.close()

    def _headers(self) -> dict[str, str]:
        headers = {
            "Content-Type": "application/json",
            "Accept": "application/json, text/event-stream",
        }
        if self.session_id:
            headers["mcp-session-id"] = self.session_id
        return headers

    @staticmethod
    def _parse_sse_or_json(response_text: str) -> dict[str, Any] | None:
        text = response_text.strip()
        if not text:
            return None

        if text.startswith("{"):
            return json.loads(text)

        for line in text.splitlines():
            if line.startswith("data:"):
                payload = line.split("data:", 1)[1].strip()
                if payload:
                    return json.loads(payload)

        raise ValueError(f"Could not parse response body: {text[:300]}")

    def _post(self, payload: dict[str, Any], expect_response: bool = True) -> dict[str, Any] | None:
        response = self.client.post(self.endpoint, headers=self._headers(), json=payload)

        sid = response.headers.get("mcp-session-id")
        if sid:
            self.session_id = sid

        if response.status_code >= 400:
            raise RuntimeError(f"HTTP {response.status_code}: {response.text}")

        if not expect_response or response.status_code == 202:
            return None

        return self._parse_sse_or_json(response.text)

    def initialize(self) -> dict[str, Any]:
        payload = {
            "jsonrpc": "2.0",
            "id": 1,
            "method": "initialize",
            "params": {
                "protocolVersion": DEFAULT_PROTOCOL_VERSION,
                "capabilities": {},
                "clientInfo": {"name": "mcp-notebook", "version": "1.0.0"},
            },
        }
        response = self._post(payload, expect_response=True)
        if not response or "result" not in response:
            raise RuntimeError(f"Invalid initialize response: {response}")
        return response

    def notify_initialized(self) -> None:
        payload = {
            "jsonrpc": "2.0",
            "method": "notifications/initialized",
            "params": {},
        }
        self._post(payload, expect_response=False)

    def list_tools(self) -> list[dict[str, Any]]:
        payload = {
            "jsonrpc": "2.0",
            "id": 2,
            "method": "tools/list",
            "params": {},
        }
        response = self._post(payload, expect_response=True)
        if not response:
            raise RuntimeError("Missing tools/list response")
        return response.get("result", {}).get("tools", [])

    def read_resource(self, uri: str) -> dict[str, Any]:
        payload = {
            "jsonrpc": "2.0",
            "id": 3,
            "method": "resources/read",
            "params": {"uri": uri},
        }
        response = self._post(payload, expect_response=True)
        if not response:
            raise RuntimeError("Missing resources/read response")
        return response

    def call_tool(self, tool_name: str, arguments: dict[str, Any]) -> dict[str, Any]:
        payload = {
            "jsonrpc": "2.0",
            "id": 4,
            "method": "tools/call",
            "params": {
                "name": tool_name,
                "arguments": arguments,
            },
        }
        response = self._post(payload, expect_response=True)
        if not response:
            raise RuntimeError("Missing tools/call response")
        return response

def spawn_server(backend: str = "llm_mock") -> subprocess.Popen[str]:
    code = (
        "import os\n"
        "from skg_mcp.server import create_mcp_server\n"
        "backend = None\n"
        "mode = os.getenv('SKG_BACKEND', '').strip().lower()\n"
        "if mode in {'llm_mock', 'mock_llm'}:\n"
        "    from skg_mcp.mock_backend import LLMMockBackend\n"
        "    backend = LLMMockBackend.from_env()\n"
        "create_mcp_server(backend=backend).run(transport='streamable-http')\n"
    )

    env = os.environ.copy()
    env["SKG_BACKEND"] = backend

    return subprocess.Popen(
        ["uv", "run", "python", "-c", code],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )

def wait_for_server(endpoint: str, timeout_seconds: float = 20.0) -> None:
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        try:
            with httpx.Client(timeout=1.0) as c:
                c.get(endpoint)
            return
        except Exception:
            time.sleep(0.3)
    raise TimeoutError(f"Server did not become ready at {endpoint}")

def run_smoke_tests(endpoint: str, tool_name: str, query: str) -> list[TestResult]:
    results: list[TestResult] = []
    client = MCPStreamableHTTPClient(endpoint=endpoint)

    try:
        init = client.initialize()
        results.append(TestResult(
            name="initialize",
            passed=client.session_id is not None,
            details=f"session_id={client.session_id}, server={init.get('result', {}).get('serverInfo', {})}",
        ))

        client.notify_initialized()
        results.append(TestResult(name="notifications/initialized", passed=True, details="sent"))

        tools = client.list_tools()
        tool_names = [t.get("name") for t in tools]
        print(f"Available tools: {tool_names}")
        results.append(TestResult(
            name="tools/list",
            passed=len(tools) > 0,
            details=f"count={len(tools)}, contains_{tool_name}={tool_name in tool_names}",
        ))

        catalog = client.read_resource("skg://catalog/tools")
        results.append(TestResult(
            name="resources/read skg://catalog/tools",
            passed="result" in catalog,
            details="ok",
        ))

        call = client.call_tool(
            tool_name,
            {
                "args": {
                    "query": query,
                    "node_types": ["concept", "statement"],
                    "limit": 4,
                }
            },
        )
        results.append(TestResult(
            name=f"tools/call {tool_name}",
            passed=("result" in call and not call.get("result", {}).get("isError", False)),
            details=str(call.get("result", {}).get("structuredContent", {}))[:240],
        ))

    except Exception as exc:
        results.append(TestResult(name="smoke_test_exception", passed=False, details=repr(exc)))
    finally:
        client.close()

    return results

def print_report(results: list[TestResult]) -> int:
    print("\nMCP Smoke Test Report")
    print("=" * 60)
    failed = 0
    for r in results:
        status = "PASS" if r.passed else "FAIL"
        print(f"[{status}] {r.name}")
        print(f"  {r.details}")
        if not r.passed:
            failed += 1
    print("=" * 60)
    print(f"Total: {len(results)}, Failed: {failed}")
    return failed


In [3]:
# Config
ENDPOINT = DEFAULT_ENDPOINT
SPAWN_SERVER = True
BACKEND = "llm_mock"
TOOL_NAME = "lexical_search"
QUERY = "attention"


In [4]:
# Start local server (optional)
server_proc = None
if SPAWN_SERVER:
    server_proc = spawn_server(BACKEND)
    wait_for_server(ENDPOINT)
    print(f"Server started for backend={BACKEND} at {ENDPOINT}")
else:
    print("Using external MCP server")


Server started for backend=llm_mock at http://127.0.0.1:8000/mcp


In [5]:
results = run_smoke_tests(endpoint=ENDPOINT, tool_name=TOOL_NAME, query=QUERY)
failed = print_report(results)
if failed:
    raise RuntimeError(f"Smoke tests failed: {failed}")


Available tools: ['filter_papers', 'lexical_search', 'semantic_search', 'semantic_constraint_search', 'resolve_concept_reference', 'expand_context', 'expand_neighbors', 'get_attribution', 'get_provenance']

MCP Smoke Test Report
[PASS] initialize
  session_id=6ee056eb8e044e4d9fcd9de5865d3af5, server={'name': 'Scholarly Knowledge Graph MCP', 'version': '1.26.0'}
[PASS] notifications/initialized
  sent
[PASS] tools/list
  count=9, contains_lexical_search=True
[PASS] resources/read skg://catalog/tools
  ok
[PASS] tools/call lexical_search
  {'concepts': [{'id': 'c-lex-1', 'label': 'attention', 'aliases': None, 'concept_type': 'mock_concept', 'is_canonical': True, 'paper_id': None, 'score': 0.99, 'summary': "Synthetic concept generated for 'attention'.", 'provenance': None}, {'
Total: 5, Failed: 0


In [ ]:
# Cleanup spawned server
if server_proc is not None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=5)
    except Exception:
        server_proc.kill()
    print("Spawned server stopped")
else:
    print("No spawned server to stop")
